# Neural Network Fundamentals for Time Series

Before diving into recurrent architectures purpose-built for sequences,
we need a solid understanding of the building blocks that *all* neural
networks share: layers, activations, loss functions, and optimizers.

This notebook provides a self-contained introduction (or refresher) to
these fundamentals, then shows how a plain **Multi-Layer Perceptron (MLP)**
can already be applied to time series forecasting by converting the
problem into supervised learning with windowed lag features.

We use the **Keras 3** API throughout -- the modern, backend-agnostic
interface to building and training neural networks.

**Contents:**
1. Dense (fully connected) layers
2. Activation functions: ReLU, sigmoid, tanh
3. Loss functions for regression: MSE, MAE
4. Optimizers: SGD, Adam
5. The Keras 3 API: building, compiling, and training models
6. Data preparation: creating supervised learning datasets from time series
7. MLP for time series forecasting
8. Evaluating and visualizing predictions
9. Effect of window size
10. Limitations of MLPs for sequences

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras import layers

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

# Reproducibility
np.random.seed(42)
keras.utils.set_random_seed(42)

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

---
## 1. Dense (Fully Connected) Layers

The fundamental building block of neural networks is the **dense layer**
(also called *fully connected* or *linear* layer).  Each neuron in a dense
layer computes a weighted sum of all its inputs plus a bias, then applies
an activation function:

$$
\mathbf{h} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})
$$

where:
- $\mathbf{x} \in \mathbb{R}^n$ is the input vector
- $\mathbf{W} \in \mathbb{R}^{m \times n}$ is the weight matrix
- $\mathbf{b} \in \mathbb{R}^m$ is the bias vector
- $\sigma(\cdot)$ is a non-linear activation function
- $\mathbf{h} \in \mathbb{R}^m$ is the output (hidden representation)

A neural network is simply a *composition* of these layers: the output
of one layer becomes the input of the next.

In [ ]:
# Demonstrate a dense layer computation manually
x = np.array([1.0, 2.0, 3.0])  # 3 input features

# Weight matrix: 2 neurons, 3 inputs
W = np.array([[0.5, -0.3, 0.8],
              [0.2,  0.7, -0.1]])
b = np.array([0.1, -0.2])

# Linear transformation: z = Wx + b
z = W @ x + b
print(f"Input x:       {x}")
print(f"Linear z=Wx+b: {z}")

# Apply ReLU activation: max(0, z)
h = np.maximum(0, z)
print(f"After ReLU:    {h}")
print()
print("This is exactly what a Dense(2, activation='relu') layer does.")

In [ ]:
# The same computation using Keras
dense_layer = layers.Dense(2, activation="relu")

# Keras expects batched input: shape (batch_size, features)
x_batched = np.array([[1.0, 2.0, 3.0]])  # batch of 1
output = dense_layer(x_batched)

print(f"Keras output shape: {output.shape}")
print(f"Keras output:       {output.numpy()}")
print(f"\nNumber of parameters: {dense_layer.count_params()}")
print(f"  Weights: {dense_layer.kernel.shape} = {np.prod(dense_layer.kernel.shape)} params")
print(f"  Biases:  {dense_layer.bias.shape}   = {np.prod(dense_layer.bias.shape)} params")

---
## 2. Activation Functions

Without activation functions, stacking dense layers would just be
multiplying matrices -- the whole network would collapse into a single
linear transformation.  Non-linear activations let neural networks
approximate *any* continuous function (Universal Approximation Theorem).

### Common activation functions

| Function | Formula | Range | Use case |
|----------|---------|-------|----------|
| **ReLU** | $\max(0, x)$ | $[0, \infty)$ | Hidden layers (default) |
| **Sigmoid** | $\frac{1}{1 + e^{-x}}$ | $(0, 1)$ | Binary output, gates |
| **Tanh** | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $(-1, 1)$ | RNN hidden states |
| **Linear** | $x$ | $(-\infty, \infty)$ | Regression output layer |

In [ ]:
# Visualize activation functions
x_vals = np.linspace(-5, 5, 300)

activations = {
    "ReLU: max(0, x)": np.maximum(0, x_vals),
    "Sigmoid: 1/(1+e^{-x})": 1 / (1 + np.exp(-x_vals)),
    "Tanh": np.tanh(x_vals),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ["tab:blue", "tab:orange", "tab:green"]

for ax, (name, vals), color in zip(axes, activations.items(), colors):
    ax.plot(x_vals, vals, color=color, linewidth=2)
    ax.axhline(0, color="grey", linestyle="--", alpha=0.4)
    ax.axvline(0, color="grey", linestyle="--", alpha=0.4)
    ax.set_title(name)
    ax.set_xlabel("x")
    ax.set_ylabel("$\\sigma(x)$")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("ReLU is the default choice for hidden layers -- simple and effective.")
print("Sigmoid and tanh are used inside LSTM gates (Chapter 12, notebook 02).")

---
## 3. Loss Functions for Regression

A **loss function** measures how far the model's predictions are from the
true values.  Training adjusts the weights to *minimize* the loss.

For time series forecasting (regression), the two most common losses are:

### Mean Squared Error (MSE)

$$
\text{MSE} = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2
$$

Penalizes large errors quadratically -- a single big miss is worse than
many small misses.  This is the **default for most regression tasks**.

### Mean Absolute Error (MAE)

$$
\text{MAE} = \frac{1}{N}\sum_{i=1}^{N}|y_i - \hat{y}_i|
$$

Penalizes all errors equally.  More robust to outliers but has
a non-smooth gradient at zero.

In [ ]:
# Visualize MSE vs MAE loss surfaces
errors = np.linspace(-4, 4, 200)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(errors, errors ** 2, label="MSE ($e^2$)", linewidth=2)
ax.plot(errors, np.abs(errors), label="MAE ($|e|$)", linewidth=2)
ax.set_xlabel("Prediction error ($y - \\hat{y}$)")
ax.set_ylabel("Loss")
ax.set_title("MSE vs MAE Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("MSE penalizes large errors much more heavily than MAE.")
print("For time series: use MSE unless you have outliers.")

---
## 4. Optimizers: SGD and Adam

**Optimizers** update the model weights to minimize the loss.  They use
the *gradient* of the loss with respect to each weight (computed via
backpropagation) to decide how to adjust weights.

### Stochastic Gradient Descent (SGD)

The simplest optimizer.  Updates each weight by:

$$
w \leftarrow w - \eta \frac{\partial L}{\partial w}
$$

where $\eta$ is the **learning rate**.  Simple but can be slow and
sensitive to the learning rate.

### Adam (Adaptive Moment Estimation)

Adam maintains a per-weight *running average* of both the gradient and
its square, adapting the effective learning rate for each parameter:

$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t \qquad \text{(first moment)}
$$
$$
v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 \qquad \text{(second moment)}
$$
$$
w \leftarrow w - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

Adam is the **default choice** for most deep learning tasks.  It converges
faster and is less sensitive to the learning rate than SGD.

In [ ]:
# Keras optimizer objects
sgd = keras.optimizers.SGD(learning_rate=0.01)
adam = keras.optimizers.Adam(learning_rate=0.001)

print("SGD config:", sgd.get_config())
print()
print("Adam config:", adam.get_config())
print()
print("Adam defaults: lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-7")
print("These defaults work well for most problems -- rarely need tuning.")

---
## 5. The Keras 3 API

Keras 3 is a **backend-agnostic** deep learning API.  It can run on
TensorFlow, JAX, or PyTorch -- same code, different engines.

The workflow:
1. **Define** the model architecture (Sequential or Functional API)
2. **Compile** with an optimizer and loss function
3. **Fit** (train) on data
4. **Evaluate** on test data
5. **Predict** on new data

In [ ]:
# Build a simple MLP with the Sequential API
model_demo = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(10,)),  # input: 10 features
    layers.Dense(32, activation="relu"),                      # hidden layer
    layers.Dense(1),                                          # output: 1 value (regression)
])

model_demo.summary()

In [ ]:
# Compile: specify optimizer, loss, and optional metrics
model_demo.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"],
)

print("Model compiled with:")
print(f"  Optimizer: {model_demo.optimizer.__class__.__name__}")
print(f"  Loss:      {model_demo.loss}")
print(f"  Metrics:   {[m.name for m in model_demo.metrics]}")

In [ ]:
# Quick demonstration with random data
X_demo = np.random.randn(200, 10)
y_demo = X_demo @ np.random.randn(10) + np.random.randn(200) * 0.1

history_demo = model_demo.fit(
    X_demo, y_demo,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=0,
)

print(f"Final training loss:   {history_demo.history['loss'][-1]:.4f}")
print(f"Final validation loss: {history_demo.history['val_loss'][-1]:.4f}")

In [ ]:
# Plot learning curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_demo.history["loss"], label="Training loss")
ax.plot(history_demo.history["val_loss"], label="Validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Learning Curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Both curves should decrease together.")
print("If validation loss rises while training loss falls => overfitting.")

---
## 6. Data Preparation: Time Series to Supervised Learning

Standard neural networks (MLPs) expect tabular input: a fixed-size
feature vector $\mathbf{x}$ and a target $y$.  Time series is a
sequence, not a table.

The key idea: use a **sliding window** to create (input, target) pairs.
Given a window size $w$, we use the previous $w$ observations as features
and the next observation as the target.

$$
\mathbf{x}_t = [y_{t-w}, y_{t-w+1}, \ldots, y_{t-1}], \qquad \hat{y}_t = f(\mathbf{x}_t)
$$

In [ ]:
def create_sequences(data, window_size):
    """Convert a 1-D time series into (X, y) pairs for supervised learning.

    Parameters
    ----------
    data : array-like, shape (n_samples,)
        The time series values.
    window_size : int
        Number of past observations to use as input features.

    Returns
    -------
    X : ndarray, shape (n_samples - window_size, window_size)
        Input feature matrix.
    y : ndarray, shape (n_samples - window_size,)
        Target values.
    """
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

In [ ]:
# Demonstrate the windowing operation on a small example
example_series = np.array([10, 20, 30, 40, 50, 60, 70, 80])
window = 3

X_ex, y_ex = create_sequences(example_series, window)

print(f"Original series: {example_series}")
print(f"Window size: {window}")
print()
print("Resulting X (inputs) and y (targets):")
for i in range(len(X_ex)):
    print(f"  X[{i}] = {X_ex[i]}  ->  y[{i}] = {y_ex[i]}")
print()
print(f"X shape: {X_ex.shape}")
print(f"y shape: {y_ex.shape}")

---
## 7. Load and Explore the Data

We use the **Daily Total Female Births** dataset -- 365 days of daily
birth counts in California, 1959.  A good starter dataset: short,
univariate, no strong trend or seasonality.

In [ ]:
births = pd.read_csv(DATA_DIR / "DailyTotalFemaleBirths.csv")
print(f"Shape: {births.shape}")
print(f"Columns: {births.columns.tolist()}")
births.head()

In [ ]:
births["Date"] = pd.to_datetime(births["Date"])
births = births.set_index("Date")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(births.index, births["Births"], linewidth=0.8)
ax.set_title("Daily Total Female Births -- California, 1959")
ax.set_xlabel("Date")
ax.set_ylabel("Number of Births")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Date range: {births.index.min().date()} to {births.index.max().date()}")
print(f"Mean: {births['Births'].mean():.1f}, Std: {births['Births'].std():.1f}")

---
## 8. MLP for Time Series Forecasting

Now we put it all together:
1. Extract the raw values as a numpy array
2. Create windowed (X, y) pairs
3. Split into train/test
4. Build, compile, and train an MLP
5. Evaluate on the test set

In [ ]:
# Extract values and normalize
values = births["Births"].values.astype("float32")

# Normalize to [0, 1] for better training
v_min, v_max = values.min(), values.max()
values_scaled = (values - v_min) / (v_max - v_min)

print(f"Original range: [{v_min}, {v_max}]")
print(f"Scaled range:   [{values_scaled.min():.2f}, {values_scaled.max():.2f}]")

In [ ]:
# Create sequences with window_size = 7 (one week of history)
WINDOW_SIZE = 7
X, y = create_sequences(values_scaled, WINDOW_SIZE)

print(f"Total samples: {len(X)}")
print(f"X shape: {X.shape}  (samples, features)")
print(f"y shape: {y.shape}")

In [ ]:
# Train/test split -- use the last 60 days as test
n_test = 60
X_train, X_test = X[:-n_test], X[-n_test:]
y_train, y_test = y[:-n_test], y[-n_test:]

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

In [ ]:
# Build the MLP
model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(WINDOW_SIZE,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1),  # linear output for regression
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    verbose=0,
)

print(f"Final training loss (MSE):   {history.history['loss'][-1]:.6f}")
print(f"Final validation loss (MSE): {history.history['val_loss'][-1]:.6f}")

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history["loss"], label="Train")
axes[0].plot(history.history["val_loss"], label="Validation")
axes[0].set_title("Loss (MSE)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history["mae"], label="Train")
axes[1].plot(history.history["val_mae"], label="Validation")
axes[1].set_title("Mean Absolute Error")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Evaluating and Visualizing Predictions

In [ ]:
# Generate predictions on the test set
y_pred_scaled = model.predict(X_test, verbose=0).flatten()

# Inverse transform to original scale
y_pred = y_pred_scaled * (v_max - v_min) + v_min
y_actual = y_test * (v_max - v_min) + v_min

# Metrics on original scale
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

print(f"Test MAE:  {mae:.2f} births")
print(f"Test RMSE: {rmse:.2f} births")

In [ ]:
# Visualize predictions vs actuals
test_dates = births.index[-n_test:]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(births.index, births["Births"], color="tab:blue", alpha=0.4, label="Full series")
ax.plot(test_dates, y_actual, color="tab:green", marker="o", markersize=3, label="Test actual")
ax.plot(test_dates, y_pred, color="tab:red", linestyle="--", label=f"MLP forecast (MAE={mae:.1f})")
ax.axvline(test_dates[0], color="grey", linestyle=":", alpha=0.6, label="Train/test split")
ax.set_title(f"MLP Forecast -- Window Size = {WINDOW_SIZE}")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Births")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis
residuals = y_actual - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(test_dates, residuals, marker="o", markersize=3, linewidth=0.8)
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0].set_title("Residuals Over Time")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Residual")
axes[0].grid(True, alpha=0.3)

axes[1].hist(residuals, bins=15, edgecolor="black", alpha=0.7)
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title(f"Residual Distribution (mean={residuals.mean():.2f})")
axes[1].set_xlabel("Residual")

plt.tight_layout()
plt.show()

---
## 9. Effect of Window Size

The window size is a critical hyperparameter.  Too small and the model
cannot capture longer patterns.  Too large and we lose training samples
and add noise.

Let us compare several window sizes.

In [ ]:
window_sizes = [3, 7, 14, 30]
results = []

for ws in window_sizes:
    X_ws, y_ws = create_sequences(values_scaled, ws)
    X_tr, X_te = X_ws[:-n_test], X_ws[-n_test:]
    y_tr, y_te = y_ws[:-n_test], y_ws[-n_test:]

    m = keras.Sequential([
        layers.Dense(64, activation="relu", input_shape=(ws,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ])
    m.compile(optimizer="adam", loss="mse")
    m.fit(X_tr, y_tr, epochs=100, batch_size=16, verbose=0, validation_split=0.15)

    preds = m.predict(X_te, verbose=0).flatten()
    preds_orig = preds * (v_max - v_min) + v_min
    actual_orig = y_te * (v_max - v_min) + v_min

    mae_ws = mean_absolute_error(actual_orig, preds_orig)
    rmse_ws = np.sqrt(mean_squared_error(actual_orig, preds_orig))
    results.append({"window": ws, "MAE": mae_ws, "RMSE": rmse_ws})
    print(f"Window={ws:2d}:  MAE={mae_ws:.2f}  RMSE={rmse_ws:.2f}")

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Visualize window size effect
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(w) for w in results_df["window"]], results_df["MAE"],
       color="tab:blue", alpha=0.7, edgecolor="black")
ax.set_xlabel("Window Size")
ax.set_ylabel("Test MAE (births)")
ax.set_title("Effect of Window Size on MLP Forecast Accuracy")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("The optimal window size depends on the data's temporal structure.")
print("For daily births with no strong weekly pattern, a window of 7-14 is typical.")

---
## 10. Limitations of MLPs for Sequences

MLPs treat the window as a *flat* feature vector.  This has important
limitations:

| Limitation | Explanation |
|-----------|-------------|
| **Fixed window** | Must choose the window size in advance; cannot adapt |
| **No parameter sharing** | Each position in the window has its own weights -- a pattern at position 1 must be re-learned at position 5 |
| **No order awareness** | The MLP treats the window as an unordered set of features; the temporal order is not explicitly encoded |
| **No memory** | Cannot carry information across windows (each prediction is independent) |

These limitations motivate **recurrent neural networks (RNNs)** and
**LSTMs**, which are *designed* to process sequences:
- They process inputs one step at a time
- They maintain a *hidden state* that carries information forward
- They share parameters across time steps

We explore these architectures in the next notebook.

---
## Summary

- **Dense layers** compute $\mathbf{h} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})$

- **Activation functions**: ReLU (hidden layers), sigmoid/tanh (gates), linear (regression output)

- **Loss functions**: MSE (default for regression), MAE (robust to outliers)

- **Optimizers**: Adam is the default choice (adaptive learning rate)

- **Keras 3 workflow**: define $\to$ compile $\to$ fit $\to$ evaluate $\to$ predict

- **Time series to supervised learning**: use a sliding window of size $w$ to
  create (X, y) pairs: $\mathbf{x}_t = [y_{t-w}, \ldots, y_{t-1}]$, target $= y_t$

- An **MLP** can forecast time series using windowed features, but it does
  not share parameters across positions or maintain memory -- motivating
  recurrent architectures (next notebook)

In [ ]:
print("Next notebook: RNN and LSTM architectures for sequence modeling.")